# 05. Avaliar atribuição — mãe + endereço

Depois do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
(modelo `p ≥ 0,90` + resgate R1/R2/R3).

Avalia notas, qualidade, quem ficou de fora, amostra por faixa/regra e
números com lista ouro.

Entradas:
- `splink_predictions_mae_endereco.parquet` ([`02d`](02d_aplicar_splink_mae_endereco.ipynb))
- `splink_atribuicao_mae_endereco.parquet` ([`04`](04_atribuir_mae_endereco.ipynb))


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    get_connection,
    lista_ouro_colunas,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
require_input(
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    label='SPLINK_ATRIBUICAO_MAE_ENDERECO (rode o 04_atribuir_mae_endereco antes)',
)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print(f'Faixas 04: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | baixo>={PISO_BAIXO} | teto={TETO}')


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

In [2]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')

# Nome distinto da TABLE atribuicao do 04 no mesmo DuckDB.
for _nome in ('atribuicao_aval', 'atribuicao_2', 'atribuicao'):
    _t = con.execute(f'''
    SELECT table_type
    FROM information_schema.tables
    WHERE table_schema = 'main' AND table_name = '{_nome}'
    ''').fetchone()
    if _t:
        _k = 'VIEW' if _t[0].upper() == 'VIEW' else 'TABLE'
        con.execute(f'DROP {_k} IF EXISTS {_nome}')

cols_atr = [
    r[0]
    for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}') LIMIT 0"
    ).fetchall()
]
tem_faixa = 'faixa' in cols_atr
tem_regra = 'regra' in cols_atr
col_faixa = 'faixa' if tem_faixa else 'CAST(NULL AS VARCHAR) AS faixa'
col_regra = 'regra' if tem_regra else 'CAST(NULL AS VARCHAR) AS regra'

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_aval AS
SELECT
    unique_id_censo,
    unique_id_cpf,
    match_probability,
    {col_faixa},
    {col_regra}
FROM read_parquet('{SPLINK_ATRIBUICAO_MAE_ENDERECO}')
''')

n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_aval').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]

print(f'Aplicação: {n_app:,}')
print(f'Atribuídos: {n_atrib:,} ({100.0 * n_atrib / n_app:.1f}%)')
print(f'Sem CPF: {n_app - n_atrib:,} ({100.0 * (n_app - n_atrib) / n_app:.1f}%)')
print(f'Pares no parquet: {n_pred:,}')
print('Colunas faixa/regra:', tem_faixa, tem_regra)


Aplicação: 6,079,030
Atribuídos: 4,310,736 (70.9%)
Sem CPF: 1,768,294 (29.1%)
Pares no parquet: 6,315,410
Colunas faixa/regra: True True


## Notas e regra dos atribuídos

Distribuição de `match_probability`, contagem por `faixa`/`regra` do 04, e
Censos por CPF.


In [3]:
display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.99 THEN 'p_ge_099'
        WHEN match_probability >= 0.90 THEN 'p_090_099'
        WHEN match_probability >= 0.75 THEN 'p_075_090'
        WHEN match_probability >= 0.50 THEN 'p_050_075'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        ELSE 'p_lt_025'
    END AS faixa_p,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval
GROUP BY 1
ORDER BY 1 DESC
''').df())

display(con.execute('''
SELECT
    coalesce(faixa, '(sem)') AS faixa_atr,
    coalesce(regra, '(sem)') AS regra_atr,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval
GROUP BY coalesce(faixa, '(sem)'), coalesce(regra, '(sem)')
ORDER BY
    CASE coalesce(faixa, '(sem)')
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END,
    coalesce(regra, '(sem)')
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao_aval
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())

,faixa_p,n,pct
0,p_ge_099,3832798,88.9
1,p_090_099,297222,6.9
2,p_075_090,45766,1.1
3,p_050_075,57525,1.3
4,p_025_050,77425,1.8


BinderException: Binder Error: column "faixa" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(faixa)" if the exact value of "faixa" is not important.

LINE 10:     CASE coalesce(faixa, '')
                           ^

## Qualidade dos pares atribuídos

No par que saiu: mãe, endereço e distância de ano de nascimento.


In [4]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())

display(con.execute(f'''
SELECT
    CASE
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'sem_data'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) = 0 THEN 'delta_0'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 1 THEN 'delta_1'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 5 THEN 'delta_2_5'
        WHEN abs(
            year(CAST(ca.data_nascimento AS DATE))
            - year(CAST(pb.data_nascimento AS DATE))
        ) <= 10 THEN 'delta_6_10'
        ELSE 'delta_gt_10'
    END AS faixa_delta_ano,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao_aval a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
GROUP BY 1
ORDER BY 1
''').df())


,n,pct_mae_preenchida,pct_mae_igual,pct_logradouro_igual,pct_cep_igual,pct_data_igual
0,4310736,41.7,24.1,25.1,65.7,83.0


,faixa_delta_ano,n,pct
0,delta_0,3760639,87.2
1,delta_1,127964,3.0
2,delta_2_5,50131,1.2
3,delta_6_10,30480,0.7
4,delta_gt_10,12466,0.3
5,sem_data,329056,7.6


## Quem ficou de fora

Melhor par no parquet por Censo sem atribuição. Quem tinha `p ≥ 0,50` e
mesmo assim não saiu: teto ou falhou o resgate (R1/R2/R3).


In [5]:
# Mesmo desempate do 04: nota → mãe preenchida → logradouro → CEP.
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_piso AS
SELECT *
FROM melhor_qualquer
WHERE match_probability >= {PISO_RESGATE}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_aval a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.50 THEN 'p_ge_050'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        WHEN m.match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_atrib + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / {n_sem}, 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_050'),
    (2, 'p_025_050'),
    (3, 'p_005_025'),
    (4, 'p_lt_005'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())

# Censos com p >= PISO no melhor-par, mas CPF estourou o teto.
con.execute(f'''
CREATE OR REPLACE TABLE cpf_n_melhor_piso AS
SELECT unique_id_cpf, COUNT(*) AS n_censos
FROM melhor_piso
GROUP BY 1
''')

n_teto = con.execute(f'''
SELECT COUNT(*)
FROM sem_cpf_melhor s
JOIN cpf_n_melhor_piso c ON c.unique_id_cpf = s.unique_id_cpf
WHERE s.match_probability >= {PISO_RESGATE}
  AND c.n_censos > {TETO}
''').fetchone()[0]
n_alto = con.execute(f'''
SELECT COUNT(*) FROM sem_cpf_melhor
WHERE match_probability >= {PISO_RESGATE}
''').fetchone()[0]
print(f'Sem CPF com melhor p >= {PISO_RESGATE}: {n_alto:,}')
print(f'  dos quais teto (CPF com >{TETO} no melhor-par): {n_teto:,}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,faixa_p,n,pct_sem_cpf,pct_aplicacao
0,p_ge_050,119213,6.7,2.0
1,p_025_050,125912,7.1,2.1
2,p_005_025,242856,13.7,4.0
3,p_lt_005,0,0.0,0.0
4,sem_par,1280313,72.4,21.1


Sem CPF com melhor p >= 0.5: 119,213
  dos quais teto (CPF com >3 no melhor-par): 382


## Pares aleatórios por faixa / regra

Amostra nova a cada execução (`ORDER BY random()`). Ajuste `N_AMOSTRA`.


In [6]:
N_AMOSTRA = 5

cols_par = '''
    a.match_probability,
    a.faixa,
    a.regra,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ca.logradouro_norm AS logradouro_censo,
    pb.logradouro_norm AS logradouro_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    a.unique_id_censo,
    a.unique_id_cpf
'''

print('--- Por faixa de nota (atribuídos) ---')
for nome, lo, hi in [
    ('p_ge_099', 0.99, 1.01),
    ('p_090_099', 0.90, 0.99),
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.match_probability >= {lo}
      AND a.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Por regra do 04 ---')
for regra in ['score', 'R1', 'R2', 'R3']:
    print(f'\n=== regra={regra} ===')
    display(con.execute(f'''
    SELECT {cols_par}
    FROM atribuicao_aval a
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
    WHERE a.regra = '{regra}'
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())

print('\n--- Sem CPF (melhor par), faixas abaixo do modelo ---')
for nome, lo, hi in [
    ('p_050_090', 0.50, 0.90),
    ('p_025_050', 0.25, 0.50),
    ('p_005_025', 0.05, 0.25),
]:
    print(f'\n=== {nome} [{lo}, {hi}) ===')
    display(con.execute(f'''
    SELECT
        s.match_probability,
        ca.nome_completo AS nome_censo,
        pb.nome_completo AS nome_cpf,
        ca.data_nascimento AS dob_censo,
        pb.data_nascimento AS dob_cpf,
        ca.nome_mae AS mae_censo,
        pb.nome_mae AS mae_cpf,
        ca.logradouro_norm AS logradouro_censo,
        pb.logradouro_norm AS logradouro_cpf,
        s.unique_id_censo,
        s.unique_id_cpf
    FROM sem_cpf_melhor s
    JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id_censo
    JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = s.unique_id_cpf
    WHERE s.match_probability >= {lo}
      AND s.match_probability < {hi}
    ORDER BY random()
    LIMIT {N_AMOSTRA}
    ''').df())


--- Por faixa de nota (atribuídos) ---

=== p_ge_099 [0.99, 1.01) ===


,match_probability,faixa,regra,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,cep_censo,cep_cpf,unique_id_censo,unique_id_cpf
0,0.999367,modelo,score,CLEONICE SANTOS ARAUJO,CLEONICE SANTOS ARAUJO,1969-08-12,1969-08-12,None,EMILIA SANT OS ARAUJO,SANTO ANTONIO,SANTO ANTONIO,None,65052180,censo_2111201100004010140120000090000001000000001,cpf_72393092315
1,0.999954,modelo,score,MARIA GOMES SOUSA,MARIA JANGARLENE GOMES SOUSA,1986-01-20,1986-01-20,None,HILTA MARIA GOMES SOUSA,AMAZONAS,AMAZONAS,65920000,65920000,censo_2111532050000170080030000175000002000000002,cpf_01263785336
2,0.999985,modelo,score,VALDEMIR MATOS,WALDEMIR MATOS FERREIRA,1942-01-21,1942-01-21,None,JUSTINA MATOS FERREIRA,DA SERRARIA,DA SERRARIA,65350000,65350000,censo_2112902050000250060010000217000001000000001,cpf_79808450300
3,0.999863,modelo,score,JOCIANA ROSA FERREIRA,JUCIANA ROSA FERREIRA,1983-01-13,1983-01-13,None,FABRICIANA ROZA FERREIRA,ITAPERA,POV ITAPERA,65250000,65250000,censo_2100204100000160000020000015000001000000002,cpf_04003460367
4,0.999999,modelo,score,MARIA ARICIMAR SOUSA BEATO,MARIA ARICIMAR SOUSA BEATO,1978-03-17,1978-03-17,None,MARIA SOUSA BEATO,11,11 QD 21,65700000,65700000,censo_2101202050001400060010000223000002000000001,cpf_00868527360



=== p_090_099 [0.9, 0.99) ===


,match_probability,faixa,regra,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,cep_censo,cep_cpf,unique_id_censo,unique_id_cpf
0,0.954046,modelo,score,FRANCISCO LOPES SILVA,FRANCISCO LOPES SILVA,1964-12-22,1964-12-22,None,MARIA LOPES SILVA,BARRO VERMELHO VIA BOCA DA MATA,PRIMEIRO DE AGOSTO,65600010,65606070,censo_2103000050001050000140000120000001000000001,cpf_39809994320
1,0.952661,modelo,score,ALIANE SILVA BARBOSA,ELIANE SILVA BARBOSA,2001-08-28,2001-08-28,None,IRACIR PEREIRA SILVA,SAO JOSE 2,S JOSE,None,65058356,censo_2111300050012450340010000344000001000000004,cpf_61079851313
2,0.971286,modelo,score,DUCILENE GOMES,LUCILENE GOMES,1967-02-05,1967-02-05,None,TEODORA GOMES,CINZA,14 DE MAIO,65210000,65055000,censo_2102507050000260000070000280000002000000002,cpf_02824941383
3,0.951257,modelo,score,EDVAN PIRES CRUZ,EDIVAN PIRES CRUZ,None,1968-01-06,None,IZABEL PIRES CRUZ,JOSE BURNET,DR JOSE BOURNET,65390000,65390000,censo_2110005050000080020020000462000001000000003,cpf_87192020320
4,0.905476,modelo,score,AURICELIA SILVA MARQUES,AURICELIA SILVA MARQUES,2001-05-26,2001-04-26,ADENORA SILVA MARQUES,ALDENORA SANTOS SILVA MARQUES,MA 119,MATOES,65727000,65727000,censo_2112233050000210000590000191000001000000003,cpf_62219112306



=== p_050_090 [0.5, 0.9) ===


...

,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.609566,BENEDITA ARAUJO,BENEDITA ARAUJO,None,1942-08-15,None,MARTINHA GENEROSA ARAUJO,ENTRE RIOS,POVOADO SAO RUMAO,censo_2109270050000460000030000308000001000000001,cpf_60031959326
1,0.799282,MACIO MAGALHAES SILVA,MARCIANO MAGALHAES SILVA,1991-08-15,1991-08-15,MARIA SEVERA MAGALHAES,MARIA SEVERO MAGALHAES,LAGOA DO FORNO A BURITIRANA,POV BURITIRANA,censo_2106607050000270000300000242000001000000003,cpf_07904756340
2,0.746727,CASTIEL PREDOSA LIMA,CASTIEL PEDROSA LIMA,2019-03-09,2019-03-09,None,PAULA VICTORIA ELOI LUZ PEDROSA,DAS PATATIVAS,DAS SIRIEMAS,censo_2111300050011390170060000185000002000000004,cpf_10397974310
3,0.527651,CARLOS ALVES NASCIMENTO,JOAO CARLOS ALVES NASCIMENTO,1995-12-08,1995-12-08,None,SOLANGE RODRIGUES ALVES,RAIMUNDO BANDEIRA BARROS,SAO JOSE,censo_2105302050001550120010000623000001000000001,cpf_61720122318
4,0.811195,LEONORA DINIZ ALMEIDA,LEONORA DINIZ ALMEIDA NASCIMENTO,None,1983-02-21,None,HOZANA AZEVEDO DINIZ,COELHO NETO,COELHO NETO,censo_2103703050000760020040000035000001000000001,cpf_00406303312



=== p_025_050 [0.25, 0.5) ===


,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.289340,LUENEIDE GOMES BARROS,LUENEIDE GOMES BARROS,None,1980-11-07,None,MARIA ROSA GOMES BARROS,1,POVOADO CAMAPUTIUA,censo_2108504050000110010050000080000001000000001,cpf_01205760318
1,0.289340,EDIMAR TEIXEIRA COSTA,EDIMAR TEIXEIRA COSTA,None,1943-02-05,None,MARIA JESUINA COSTA,SANTO JOAO DA MATA SAO DOMINGOS,SAO JOAO,censo_2104628050000100000490000071000002000000001,cpf_04414349320
2,0.267265,POLIANA VIANA LIMA,POLIANA VIANA LIMA,1992-07-07,1992-06-07,None,DELSANIRA PAIXAO VIANA CARDOSO,LOCAL 118,H,censo_2111201100000950050040000113000001000000002,cpf_02412395322
3,0.393925,ADNAN SANTOS MATOS,ADNAM SANTOS MATOS,None,1982-07-19,None,MARIA JOSE SANTOS MATOS,B,ANTENOR MELO,censo_2111300050011580010010000133000001000000001,cpf_01365609383
4,0.289340,JOSE ROMARIO GOMES COSTA,JOSE ROMARIO GOMES COSTA,None,1994-06-21,None,MARIA CRUZ PEREIRA GOMES,13 DE ABRIL,CINCO,censo_2103000050003000150040000093000001000000002,cpf_62410956335



=== p_005_025 [0.05, 0.25) ===


,match_probability,nome_censo,nome_cpf,dob_censo,dob_cpf,mae_censo,mae_cpf,logradouro_censo,logradouro_cpf,unique_id_censo,unique_id_cpf
0,0.126641,LUCAS GABRIEL SILVA SILVA,LUCAS GABRIEL SILVA SILVA,2018-11-22,2014-11-22,None,MARIA SOCORRO SANTOS SILVA,DO CAMPO,2 TRAVESSA PRIMEIRO DE MAIO,censo_2110302050000060040020000147000001000000005,cpf_63695831383
1,0.090135,RAIMUNDO ALVES,RAIMUNDO ALVES,None,1960-06-09,None,ETELINA ALVES,FLAVIO PICANCO,TRAV FLAVIO PICANCO,censo_2103703050000140080040000066000002000000001,cpf_18652670234
2,0.143752,RAIONE PEREIRA SANTOS,RAYONE PEREIRA SANTOS,1996-08-06,1988-05-01,None,IRENILDE SILVA PEREIRA,DA RUA DO CAMPO,PRINCIPAL,censo_2106706050000420020030000101000001000000001,cpf_02025543336
3,0.190328,KELVE SILVA COSTA SILVA COSTA,KELVIN SILVA COSTA,2003-08-08,2003-08-08,None,JOSEFA REIS SILVA,SANTA MARIA,SANTA MARIA,censo_2104073050000070000060000021000001000000006,cpf_11499301367
4,0.101964,JEDEILSON SILVA LIMA,EDIELSON SILVA LIMA,2000-03-09,2000-03-09,FRANCISCA CONCEICAO SILVA,IVANETE MARIA SILVA LIMA,ANA TAIS SALES,ACUDE,censo_2105708050000160200030000371000001000000002,cpf_07583668336


## Observação sobre piso

O 04 já resgata `0,25 ≤ p < 0,90` com R1/R2/R3. Baixar o piso do **modelo**
(`p ≥ 0,90`) sem regra tende a puxar falso positivo — use a amostra acima
antes de afrouxar.


In [7]:
display(con.execute(f'''
SELECT
    coalesce(faixa, '(sem)') AS faixa,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_atrib,
    ROUND(100.0 * COUNT(*) / {n_app}, 1) AS pct_aplicacao
FROM atribuicao_aval
GROUP BY 1
ORDER BY
    CASE coalesce(faixa, '')
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
        ELSE 9
    END
''').df())


BinderException: Binder Error: column "faixa" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(faixa)" if the exact value of "faixa" is not important.

LINE 10:     CASE coalesce(faixa, '')
                           ^

## Lista ouro (censo_limpo)

Universo com a lista ainda dentro. Quem está na ouro e no pareamento conta
como `lista_ouro`. `nos_dois` tem de ser zero (essa gente saiu da aplicação
antes do 04).


In [8]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao_aval
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao_aval) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')
assert n_ambos == 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,origem,n,pct
0,lista_ouro,510774,7.8
1,pareamento,4310736,65.4
2,sem_cpf,1768294,26.8


nos_dois: 0


In [9]:
con.close()
print('Fim do 05.')


Fim do 05.
